# Day 6: Database Integration - SQLAlchemy, Models, and Sessions

## Goal
Move from in-memory data structures to persistent database using SQLAlchemy ORM, understanding relationships, constraints, and session management.

## Why This Day Matters
Most bugs in production come from database issues: N+1 queries, race conditions, forgotten commits. Mastering SQLAlchemy prevents these.

In [ ]:
from sqlalchemy import create_engine, Column, Integer, String, Float, DateTime, ForeignKey, Boolean, Table
from sqlalchemy.orm import declarative_base, sessionmaker, relationship, Session
from datetime import datetime
import os

# Create in-memory SQLite for demonstration
DATABASE_URL = "sqlite:///:memory:"
engine = create_engine(DATABASE_URL, echo=True)
Base = declarative_base()

print("✓ SQLAlchemy configured for in-memory SQLite")

## PART 1: SQLALCHEMY MODELS

SQLAlchemy models define database tables and their structure.

In [ ]:
# EXAMPLE 1: Basic model
class User(Base):
    __tablename__ = "users"
    
    id = Column(Integer, primary_key=True)
    username = Column(String(50), unique=True, nullable=False, index=True)
    email = Column(String(100), unique=True, nullable=False, index=True)
    password_hash = Column(String(255), nullable=False)
    is_active = Column(Boolean, default=True)
    created_at = Column(DateTime, default=datetime.utcnow)
    
    def __repr__(self):
        return f"<User(id={self.id}, username='{self.username}')>"

# Map: Column types to SQL types
print("✓ SQLAlchemy model structure:")
print(f"  id: Integer (primary key)")
print(f"  username: String(50) (unique, indexed)")
print(f"  email: String(100) (unique, indexed)")
print(f"  password_hash: String(255)")
print(f"  is_active: Boolean (default=True)")
print(f"  created_at: DateTime (auto-timestamp)")

In [ ]:
# EXAMPLE 2: More complex model with relationships
class Product(Base):
    __tablename__ = "products"
    
    id = Column(Integer, primary_key=True)
    name = Column(String(200), nullable=False, index=True)
    description = Column(String(5000))
    price = Column(Float, nullable=False)
    stock = Column(Integer, default=0)
    created_at = Column(DateTime, default=datetime.utcnow)
    
    # For now, just know relationships exist (covered next)
    def __repr__(self):
        return f"<Product(id={self.id}, name='{self.name}', price=${self.price})>"

print("✓ Product model with pricing and inventory tracking")

## PART 2: RELATIONSHIPS (ONE-TO-MANY)

Connect tables together.

In [ ]:
# Re-define User with posts relationship
class User(Base):
    __tablename__ = "users"
    
    id = Column(Integer, primary_key=True)
    username = Column(String(50), unique=True, nullable=False)
    email = Column(String(100), unique=True, nullable=False)
    posts = relationship("Post", back_populates="author")  # Relationship

class Post(Base):
    __tablename__ = "posts"
    
    id = Column(Integer, primary_key=True)
    title = Column(String(200), nullable=False)
    content = Column(String(5000))
    author_id = Column(Integer, ForeignKey("users.id"), nullable=False)  # Foreign key
    created_at = Column(DateTime, default=datetime.utcnow)
    
    author = relationship("User", back_populates="posts")  # Back-reference

print("✓ One-to-Many: One user can have many posts")
print(f"  User.posts -> list of Post objects")
print(f"  Post.author_id -> points to users.id (foreign key)")
print(f"  Post.author -> User object (relationship)")

## PART 3: MANY-TO-MANY RELATIONSHIPS

When both sides have many.

In [ ]:
,
,
,
user_groups",
    Base.metadata,
    Column("user_id", Integer, ForeignKey("users.id")),
    Column("group_id", Integer, ForeignKey("groups.id"))
)

class Group(Base):
    __tablename__ = "groups"
    
    id = Column(Integer, primary_key=True)
    name = Column(String(100), unique=True, nullable=False)
    members = relationship("User", secondary=user_groups, backref="groups")

# Update User to include groups
# User.members -> list of Group objects
# Group.members -> list of User objects

print("✓ Many-to-Many: Users and Groups")
print(f"  - User.groups -> list of Group objects")
print(f"  - Group.members -> list of User objects")
print(f"  - Implemented via junction table 'user_groups'")
]
)

## PART 4: DATABASE OPERATIONS (CREATE TABLES)

Turn models into actual database tables.

In [ ]:
# Create all tables
Base.metadata.create_all(engine)
print("✓ All tables created in database")

# Create session factory
SessionLocal = sessionmaker(bind=engine)
print("✓ SessionLocal created for database access")

## PART 5: SESSION MANAGEMENT

Sessions are how you interact with the database.

In [ ]:
# EXAMPLE 1: Creating records
session = SessionLocal()

# Create user
user1 = User(
    username="alice",
    email="alice@example.com",
    password_hash="hashed_password_here"
)

# Add to session
session.add(user1)
# Flush writes to DB but keeps transaction open
session.flush()
print(f"✓ User created with ID: {user1.id}")

# Commit completes the transaction
session.commit()
print("✓ Transaction committed")

session.close()
print("✓ Session closed")

In [ ]:
# EXAMPLE 2: Reading records
session = SessionLocal()

# Get by primary key
user = session.query(User).filter(User.username == "alice").first()
print(f"✓ Found user: {user.username} ({user.email})")

# Get all records
all_users = session.query(User).all()
print(f"✓ Total users: {len(all_users)}")

session.close()

In [ ]:
# EXAMPLE 4: Deleting records
session = SessionLocal()

user = session.query(User).filter(User.username == "alice").first()
session.delete(user)
session.commit()
print("✓ User deleted")

session.close()

## PART 6: IMPORTANT: N+1 QUERY PROBLEM

A common performance killer.

In [ ]:
# ❌ SLOW: N+1 queries
# Assume 1000 users
session = SessionLocal()

users = session.query(User).all()  # Query 1
for user in users:
    print(user.id)  # This would load groups for each user -> 1000 more queries!
    # Accessing user.groups triggers a new query because it's lazy-loaded

session.close()

print("\n❌ This causes 1 query + 1000 queries = 1001 total queries!")

## PART 7: CONSTRAINTS AND MIGRATIONS

Keeping database changes organized.

## PART 8: SQLALCHEMY WITH FASTAPI

How to integrate SQLAlchemy into your API.

## PART 9: BEST PRACTICES

### ✅ DO:
1. Always use `session.commit()` or `session.rollback()`
2. Use `Depends(get_db)` for session management in FastAPI
3. Use eager loading (`joinedload`) for common relationships
4. Add indexes on frequently searched fields
5. Use foreign keys to maintain referential integrity
6. Use separate Pydantic models for requests/responses

### ❌ DON'T:
1. Leave sessions open (memory leak, connection pool exhaustion)
2. Access relationships without eager loading in loops
3. Expose ORM models directly in API responses
4. Modify objects without calling commit()
5. Use mutable default arguments

## Day 6 Summary

You now understand:
✅ SQLAlchemy ORM models and table structure
✅ Relationships (one-to-many, many-to-many)
✅ Session management and CRUD operations
✅ N+1 query problem and prevention
✅ Constraints and indexes
✅ Integration with FastAPI using Depends()
✅ Why separate Pydantic schemas from ORM models

**Next:** Day 7 - CRUD operations at scale (pagination, filtering, sorting)